
# FRE-GT-9743 Assignment 3 - Bond Future: Theoretical Questions

Setup: a 10-year bond with exactly 9 years to maturity, coupon $c = 4\%$ (semi-annual), spot price $S = 100$, delivery in $T_d = 3$ months ($0.25$ years), market repo $r_{mkt} = 2\%$.



## Q1. Normalized forward price

**1) 3-month forward price $BF$.**

"Exactly 9 years to maturity" with semi-annual coupons means today is a coupon date, so the next coupon falls at $t = 0.5$, which is after the delivery date $T_d = 0.25$. No coupon is paid in $(0, T_d]$, so the repo cost-of-carry formula has no coupon term:

$$
BF = S\,(1 + r_{mkt}\,T_d) - \sum_{t_j \in (0,T_d]} c_j\,\big(1 + r_{mkt}(T_d - t_j)\big) = 100\,(1 + 0.02 \times 0.25) = 100.50
$$

**2) Conversion factor $K$.**

$K$ is the clean price of the bond at a constant yield of $6\%$, divided by 100. The bond has $n = 18$ semi-annual periods left, semi-annual coupon $= 100 \times 4\%/2 = 2$, and the periodic yield is $6\%/2 = 3\%$:

$$
K = \frac{1}{100}\left[\sum_{i=1}^{18} \frac{2}{(1.03)^i} + \frac{100}{(1.03)^{18}}\right] = \frac{86.246487}{100} = 0.862465
$$

Since today is a coupon date, accrued interest is zero, so this price is both clean and dirty.

**3) Normalized forward price.**

$$
\frac{BF}{K} = \frac{100.50}{0.862465} = 116.5265
$$

**4) What does the conversion factor do?**

Bonds in the basket have different coupons and maturities, so their forward prices are not directly comparable. $K$ rescales each bond to the price it would have if it yielded the notional $6\%$, which puts every bond on the same scale. Dividing the forward price by $K$ then gives the futures-equivalent price of delivering that bond, and the bonds can be ranked.

Accrued interest: $K$ is built from the **clean** price, so it contains no accrued interest and does not depend on where the bond sits inside its coupon period. Accrued interest is handled separately. The short receives the invoice amount
$$
F \times K + \text{AI},
$$
so the accrued interest is added on top of $F \times K$ and does not enter the conversion factor.


In [9]:
# Verify Q1 numerically
S = 100.0
coupon_rate = 0.04
freq = 2
n = 18                      # 9 years * 2
y = 0.06                    # conversion-factor yield
r_mkt = 0.02
Td = 0.25

# 1) forward price (no coupon between 0 and Td since next coupon is at t = 0.5)
BF = S * (1 + r_mkt * Td)

# 2) conversion factor
c = 100 * coupon_rate / freq
per_y = y / freq
price_6 = sum(c / (1 + per_y) ** i for i in range(1, n + 1)) + 100 / (1 + per_y) ** n
K = price_6 / 100

# 3) normalized forward
norm_fwd = BF / K

print(f"BF        = {BF:.6f}")
print(f"Price@6%  = {price_6:.6f}")
print(f"K         = {K:.6f}")
print(f"BF / K    = {norm_fwd:.6f}")


BF        = 100.500000
Price@6%  = 86.246487
K         = 0.862465
BF / K    = 116.526485



## Q2. Cheapest-to-deliver and implied repo

The future price is
$$
F = \min_{x \in X} \frac{BF_x(S, r_x, T_d)}{K_x}.
$$

**1) Implied repo of the bond in Q1 for $F = 115.3670$.**

Back out the forward price implied by the future:
$$
BF = F \times K = 115.3670 \times 0.862465 = 99.50
$$
Solve $BF = S\,(1 + r_{imp}\,T_d)$ (no coupon in the period):
$$
r_{imp} = \frac{BF/S - 1}{T_d} = \frac{99.50/100 - 1}{0.25} = -2.00\%
$$

The implied repo is negative. This follows from the numbers given: at $F = 115.3670$ the implied forward is below the spot price. If the result looks off against what was intended, it is worth confirming the given $F$ with the instructor.

**2) Which bond is cheapest to deliver?**

The short position chooses which bond to deliver and will pick the one that is cheapest, i.e. the one with the smallest $BF_x/K_x$. A smaller $BF_x/K_x$ corresponds to a higher implied repo $r^{imp}_x$, because $BF_x$ is increasing in the repo rate.

Compare each $r^{imp}_x$ to the general collateral repo $r_{mkt}$:

- If $r^{imp}_x > r_{mkt}$: buying bond $x$, financing it at $r_{mkt}$ and delivering it into the future earns more than the funding cost. This is a profitable delivery, and the higher the gap, the cheaper the bond is to deliver.
- If $r^{imp}_x < r_{mkt}$: delivering that bond earns less than the funding cost, so the short would prefer another bond.

Decision rule: compute $r^{imp}_x$ for every bond in the basket. **The CTD is the bond with the highest implied repo** (equivalently the lowest $BF_x/K_x$). The exchange mechanism, in which the short delivers whichever bond is cheapest, is what makes the market price $F$ equal that minimum.


In [10]:
# Verify Q2.1 numerically
F = 115.3670
BF_implied = F * K
r_imp = (BF_implied / S - 1) / Td

print(f"BF implied by F = {BF_implied:.6f}")
print(f"Implied repo    = {r_imp * 100:.4f} %")


BF implied by F = 99.499985
Implied repo    = -2.0001 %



## Q3. Scenario analysis: how can the CTD change, and why does it matter?

**What can change the CTD over the next $N$ days**

The CTD is the bond with the lowest $BF_x/K_x$. Three things move it.

1. **Yield level relative to the $6\%$ notional (main driver).** $K$ prices every bond at a flat $6\%$, but a bond's market price responds to yields according to its duration, which $K$ does not capture.
   - When yields are **above 6%** (and especially when rising), shorter-duration bonds (shorter maturity, higher coupon) tend to become cheapest to deliver. Longer-duration bonds lose more market value than $K$ assumes.
   - When yields are **below 6%** (and falling), longer-duration bonds (longer maturity, lower coupon) tend to become cheapest to deliver.

2. **Curve shape.** A steepening or flattening of the curve changes the relative yields of shorter and longer bonds in the basket, so the CTD can switch even if the average yield level does not move.

3. **Carry and repo.** Each bond's coupon relative to the repo rate affects its forward price. Over $N$ days, coupon payments, accrued interest, shrinking time to delivery and any repo specialness on a particular bond change each $BF_x/K_x$.

**How to predict it:** compute $r^{imp}_x$ for each bond today. Apply a scenario to the yield curve (for example a parallel move of $\pm 25$ to $\pm 50$ bp, or a steepening), roll the forward prices $N$ days, recompute $BF_x/K_x$, and take the bond with the lowest value (highest implied repo). If two bonds have close implied repos today, a small yield move can switch the CTD.

**Why it matters which bond is delivered**

- **Pricing:** the futures price converges to $BF/K$ of the CTD, so identifying the wrong CTD gives the wrong fair value for $F$.
- **Hedging:** the future behaves like the CTD, so its duration and DV01 are those of the CTD. A hedge ratio built on the wrong bond is wrong, and if the CTD switches, the future's duration changes and the hedge no longer works.
- **Delivery option:** the short chooses which bond to deliver, so the short holds a switch option. The wider the range of plausible CTDs, the more that option is worth, and this value is embedded in the futures price.
- **Basis trading:** the spread between a bond and the future (the basis) depends directly on which bond is cheapest, so relative-value trades are sensitive to CTD changes.
